# 1. Install Modules

In [ ]:
# Bioinformatics Tools (Ubuntu)
!sudo apt-get update
!sudo apt-get install -y fastp flash bwa samtools

# Python Library
!pip3 install biopython cutadapt pysam --break-system-packages

# 2 Trimming and Discard trimmed sample

In [ ]:
import subprocess
import glob
import os

# Specify the folder containing your input files.
# Specify the folder where you want to save the untrimmed (adapter-free) sequences.
input_folder = "fastq"
untrimmed_output_folder = "fastq/stage1/A_untrimmed_output"

# Define the adapter sequences for R1 and R2.
adapter_sequence_r1 = "AGATCGGAAGAGCACACGTCTGAACTCCAGTCAC"
adapter_sequence_r2 = "AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT"

# Use glob to get a list of all input file pairs (R1 and R2) in the folder.
input_file_pairs = []
for input_r1 in glob.glob(os.path.join(input_folder, "*_R1.fastq.gz")):
    # Assuming R2 files have the same naming format as R1 files.
    input_r2 = input_r1.replace("_R1.fastq.gz", "_R2.fastq.gz")
    if os.path.exists(input_r2):  # Ensure R2 file exists.
        input_file_pairs.append({"r1": input_r1, "r2": input_r2})

# Create the output folder if it doesn't exist.
os.makedirs(untrimmed_output_folder, exist_ok=True)

for input_files in input_file_pairs:
    input_r1 = input_files["r1"]
    input_r2 = input_files["r2"]

    # Define output file paths for untrimmed (clean, adapter-free) sequences.
    untrimmed_r1 = os.path.join(untrimmed_output_folder, os.path.basename(input_r1).replace(".fastq.gz", "_untrimmed.fastq.gz"))
    untrimmed_r2 = os.path.join(untrimmed_output_folder, os.path.basename(input_r2).replace(".fastq.gz", "_untrimmed.fastq.gz"))

    # Use cutadapt to keep only untrimmed sequences (completely adapter-free).
    result = subprocess.run([
        "cutadapt",
        "-a", adapter_sequence_r1,  # Adapter for R1
        "-A", adapter_sequence_r2,  # Adapter for R2
        "-O", "15",                  # Minimum overlap for adapter trimming
        "--discard-trimmed",         # Discard sequences where trimming occurred
        "-o", untrimmed_r1,          # Save only untrimmed R1 reads
        "-p", untrimmed_r2,          # Save only untrimmed R2 reads
        input_r1, input_r2
    ], capture_output=True, text=True)

    # Log the result.
    if result.returncode == 0:
        print(f"Untrimmed sequences saved: {untrimmed_r1}, {untrimmed_r2}")
    else:
        print(f"Error processing {input_r1} and {input_r2}:\n{result.stderr}")

# 3. Q filtering

In [ ]:
import os
import subprocess

# Quality threshold (Phred score)
quality_threshold = 30

# Set input and output folders
input_folder = "fastq/stage1/A_untrimmed_output"
output_folder = "fastq/stage1/B_Qfiltered"

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Iterate through files in the input folder, processing only those ending with "_untrimmed.fastq.gz"
for filename in os.listdir(input_folder):
    if filename.endswith("_untrimmed.fastq.gz"):
        # Input file path
        input_file = os.path.join(input_folder, filename)
        
        # Output filename (e.g., sample_untrimmed.fastq.gz -> sample_Qfiltered.fastq.gz)
        output_file = os.path.join(
            output_folder, 
            filename.replace("_untrimmed.fastq.gz", "_Qfiltered.fastq.gz")
        )
        
        # Execute fastp in single-end mode for each file
        subprocess.call([
            "fastp",
            "-i", input_file,                      # Input file
            "-o", output_file,                     # Output file
            "-q", str(quality_threshold),          # Quality threshold for a base to be qualified
            "-u", "15",                            # Discard reads if the percentage of unqualified bases is >= 15%
            "-l", "151",                           # Minimum read length to keep
            "--cut_mean_quality", "30",            # Discard reads if mean quality is less than 30
            "--html", f"{output_file}.html",       # HTML report file path
            "--json", f"{output_file}.json"        # JSON report file path
        ])
        
        print(f"Filtering for {filename} is complete.\n"
              f"Output FASTQ : {output_file}\n"
              f"Reports      : {output_file}.html / {output_file}.json\n")

print("All filtering processes are done.")

# 4. Match Paired-End Read IDs

In [ ]:
import gzip
import glob
import os

def extract_matching_reads(r1_path, r2_path, out_r1_path, out_r2_path):
    def get_read_id(header):
        # Extract ID from the FASTQ header
        return header.split()[0].replace('/1', '').replace('/2', '')

    r1_ids = set()
    r2_ids = set()

    # Extract all read IDs from the R1 file
    with gzip.open(r1_path, 'rt') as r1_file:
        while True:
            header = r1_file.readline()
            if not header:
                break
            r1_ids.add(get_read_id(header.strip()))
            # Skip the other 3 lines of the read (sequence, +, quality)
            [r1_file.readline() for _ in range(3)] 

    # Extract all read IDs from the R2 file
    with gzip.open(r2_path, 'rt') as r2_file:
        while True:
            header = r2_file.readline()
            if not header:
                break
            r2_ids.add(get_read_id(header.strip()))
            [r2_file.readline() for _ in range(3)]

    # Find common and unique IDs
    matching_ids = r1_ids & r2_ids
    r1_only = r1_ids - r2_ids
    r2_only = r2_ids - r1_ids

    print(f"Processing {os.path.basename(r1_path)} and {os.path.basename(r2_path)}")
    print(f"Total R1 IDs: {len(r1_ids)}, Total R2 IDs: {len(r2_ids)}, Matching IDs: {len(matching_ids)}")
    print(f"IDs only in R1: {len(r1_only)}, IDs only in R2: {len(r2_only)}\n")

    # Create the output directory if it doesn't exist
    os.makedirs(os.path.dirname(out_r1_path), exist_ok=True)

    # Function to write only the reads with matching IDs to a new file
    def write_matching_reads(input_path, output_path, matching_ids):
        with gzip.open(input_path, 'rt') as infile, gzip.open(output_path, 'wt') as outfile:
            while True:
                lines = [infile.readline() for _ in range(4)]
                if not lines[0]:
                    break
                read_id = get_read_id(lines[0].strip())
                if read_id in matching_ids:
                    outfile.writelines(lines)

    # Write the filtered R1 and R2 files
    write_matching_reads(r1_path, out_r1_path, matching_ids)
    write_matching_reads(r2_path, out_r2_path, matching_ids)

# --------------------------
# Apply to all file pairs
# --------------------------

input_folder = "fastq/stage1/B_Qfiltered"
output_folder = "fastq/stage1/C_id_matched"

# Find all R1 files
r1_files = glob.glob(os.path.join(input_folder, "*_R1_Qfiltered.fastq.gz"))

# For each R1, find the corresponding R2 file and run the process
for r1_file in r1_files:
    r2_file = r1_file.replace("_R1_Qfiltered.fastq.gz", "_R2_Qfiltered.fastq.gz")
    
    if os.path.exists(r2_file):
        # Set the output file paths
        base_name = os.path.basename(r1_file).replace("_R1_Qfiltered.fastq.gz", "")
        out_r1 = os.path.join(output_folder, f"{base_name}_ID_match_R1.fastq.gz")
        out_r2 = os.path.join(output_folder, f"{base_name}_ID_match_R2.fastq.gz")
        
        # Execute the function
        extract_matching_reads(r1_file, r2_file, out_r1, out_r2)
    else:
        print(f"Warning: Corresponding R2 file not found for {r1_file}. Skipping.")

# 5 Merge W/ Flash

In [ ]:
import os
import glob
import subprocess

# === Folder Setup ===
# input_folder: Directory containing matched R1/R2 fastq.gz files
# output_folder: Directory to save FLASH merging results
input_folder = "fastq/stage1/C_id_matched"
output_folder = "fastq/stage1/E_random_merged_output"
os.makedirs(output_folder, exist_ok=True)

# List and sort all R1 files (*_R1.fastq.gz)
r1_files = sorted(glob.glob(os.path.join(input_folder, "*_R1.fastq.gz")))

print(f"🔎 Found {len(r1_files)} R1 files to process.")

# Fixed Overlap Configuration
MIN_OVERLAP = "117"
MAX_OVERLAP = "133"

for r1_path in r1_files:
    r1_filename = os.path.basename(r1_path)
    
    # 1. Construct the matching R2 file path
    r2_path = r1_path.replace("_R1.fastq.gz", "_R2.fastq.gz")
    
    if not os.path.exists(r2_path):
        print(f"⚠️ Matching R2 file not found for: {r1_filename} → Skipping.")
        continue

    # 2. Define output naming convention (extract tag from filename)
    sample_tag = r1_filename.replace(".fastq.gz", "").replace("_R1", "")
    output_name = f"{sample_tag}_FLASH"

    print(f"🔵 Running FLASH: {sample_tag} | Range: {MIN_OVERLAP}-{MAX_OVERLAP}")
    
    try:
        # Note: All subprocess arguments must be strings
        subprocess.check_call([
            "flash",
            "-m", MIN_OVERLAP,
            "-M", MAX_OVERLAP,
            "-o", output_name,
            "-d", output_folder,
            r1_path,
            r2_path
        ])
        print(f"✅ FLASH merging complete → {output_name}.extendedFrags.fastq")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ FLASH merging failed for {sample_tag}: {e}")

print("\n🚀 All FLASH processes finished within the specified range.")

# 6. fastq -> fasta

In [ ]:
import os
from Bio import SeqIO
from pathlib import Path

# === 1. Path Configuration ===
# input_folder: Directory containing FLASH merged results (.extendedFrags.fastq)
# output_folder: Directory where converted FASTA files will be saved
input_folder = Path("fastq/stage1/E_random_merged_output")
output_folder = Path("fastq/stage1/F_fasta_output")

os.makedirs(output_folder, exist_ok=True)

# === 2. Conversion Loop ===
# Target only 'extendedFrags.fastq' files for conversion
fastq_files = list(input_folder.glob("*.extendedFrags.fastq"))

if not fastq_files:
    print(f"⚠️  No 'extendedFrags.fastq' files found in '{input_folder}'.")
else:
    print(f"🔎 Starting conversion for {len(fastq_files)} files.")

    for fastq_path in fastq_files:
        # Generate clean output filename
        # Example: sample_FLASH.extendedFrags.fastq -> sample_FLASH.fasta
        fasta_name = fastq_path.name.replace(".extendedFrags.fastq", ".fasta")
        fasta_path = output_folder / fasta_name

        try:
            # SeqIO.convert is memory-efficient and ideal for large genomic files
            with open(fastq_path, "r") as in_handle:
                with open(fasta_path, "w") as out_handle:
                    count = SeqIO.convert(in_handle, "fastq", out_handle, "fasta")
            
            print(f"✅ Conversion Complete: {fastq_path.name} → {fasta_name} ({count} reads)")
            
        except Exception as e:
            print(f"❌ Error processing '{fastq_path.name}': {e}")

print("\n✨ All conversion tasks have been successfully completed!")

# 7. Binary data reference seqeunce data generate

In [ ]:
from pathlib import Path

def generate_sequences_for_bit(bit_length: int):
    """
    Generate DNA sequences for all binary combinations of the given bit_length.
    (bit_length=8 -> 256 barcodes)
    """
    sequences = {}

    seq_0 = "ACTCATATACACACTTAATC"
    seq_1 = "ACTCATATACATACACTTAATC"
    prefix = "ACACTTAATC"

    for i in range(2 ** bit_length):
        binary_str = format(i, f'0{bit_length}b')
        sequence = ''.join(seq_1 if bit == '1' else seq_0 for bit in binary_str)
        full_sequence = prefix + sequence
        seq_id = f"seq_{i:03d}_{binary_str}"
        sequences[seq_id] = full_sequence

    return sequences

def write_fasta(sequences: dict, output_path: str):
    """Write sequences to a FASTA file."""
    with open(output_path, "w") as f:
        for seq_id, sequence in sequences.items():
            f.write(f">{seq_id}\n{sequence}\n")

# ===== Settings: exactly 8 bits (256 barcodes) =====
BIT_LENGTH = 8  # 2^8 = 256
output_dir = Path("reference_sequence")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "8bit_reference.fasta"
# ==================================================

seqs = generate_sequences_for_bit(BIT_LENGTH)
write_fasta(seqs, output_path)
print(f"✅ 8-bit (256) barcodes FASTA saved: {output_path}")

# 8. Align

## 8.1 Reference sequence - Sample Matching

In [ ]:
# Index reference
!bwa index "reference_sequence/8bit_reference.fasta"

In [ ]:
%%bash
# Set the path to the reference sequence file
reference_file="reference_sequence/8bit_reference.fasta"

# Set the directory containing your filtered FASTA files
fasta_directory="fastq/stage1/F_fasta_output"
# Set the output directory for aligned SAM files
output_dir="fastq/stage1/1_align_sam"

# Make sure the output directory exists or create it if necessary
mkdir -p "$output_dir"

# Iterate through filtered FASTA files in the specified directory
for fasta_file in "$fasta_directory"/*_match_FLASH.fasta; do
    # Generate an output file name based on the input filename
    output_file="$output_dir/$(basename "$fasta_file" .fasta).sam"

    # Perform the BWA alignment 
    bwa mem -M -t 4 "$reference_file" "$fasta_file" > "$output_file"

    echo "Alignment completed for $fasta_file. Result saved as $output_file"
done

## 8.1 sam to bam

In [ ]:
%%bash

# Set the path to the directory containing SAM files
sam_dir="fastq/stage1/1_align_sam"
# Set the output directory for BAM files
bam_dir="fastq/stage1/2_align_bam"

# Make sure the output directory exists or create it if necessary
mkdir -p "$bam_dir"

# Convert SAM files to BAM
for sam_file in "$sam_dir"/*.sam; do
    bam_file="$bam_dir/$(basename "$sam_file" .sam).bam"
    samtools view -bS "$sam_file" -o "$bam_file"
    echo "Conversion from $sam_file to $bam_file is complete."
done

## 8.2  Convert BAM to CSV

In [ ]:
import os
import pysam
import pandas as pd

# Input folder (path where BAM files are located)
input_folder = "fastq/stage1/2_align_bam"
# Output folder (path to save CSV files)
output_folder = "fastq/stage1/3_align_bam/csv"

# Create the output folder if it does not exist
os.makedirs(output_folder, exist_ok=True)

# Function to convert a BAM file to CSV, including optional fields
def bam_to_csv(bam_file, output_folder):
    output_csv = os.path.join(output_folder, os.path.basename(bam_file).replace(".bam", ".csv"))
    
    # Read the BAM file.
    with pysam.AlignmentFile(bam_file, "rb") as bam:
        records = []
        
        for read in bam:
            # Standard BAM fields.
            record = {
                "QNAME": read.query_name,
                "FLAG": read.flag,
                "RNAME": bam.get_reference_name(read.reference_id) if read.reference_id >= 0 else "*",
                "POS": read.reference_start + 1,
                "MAPQ": read.mapping_quality,
                "CIGAR": read.cigarstring if read.cigarstring else "*",
                "RNEXT": bam.get_reference_name(read.next_reference_id) if read.next_reference_id >= 0 else "*",
                "PNEXT": read.next_reference_start + 1 if read.next_reference_start >= 0 else 0,
                "TLEN": read.template_length,
                "SEQ": read.query_sequence if read.query_sequence else "*",
                "QUAL": read.qual if read.qual else "*",
            }
            
            # Add optional fields (tags).
            for tag, value in read.tags:
                record[tag] = value

            records.append(record)
    
    # Create a DataFrame from the list of records.
    df = pd.DataFrame(records)

    # Fill any missing optional fields with "*" instead of NaN for consistency.
    df = df.fillna("*")

    # Save the DataFrame to a CSV file.
    df.to_csv(output_csv, index=False)
    print(f"Converted: {os.path.basename(bam_file)} -> {os.path.basename(output_csv)}")
    return output_csv

# Find all BAM files in the input folder.
bam_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith(".bam")]

# Convert all found BAM files to CSV.
csv_files = []
for bam_file in bam_files:
    csv_file = bam_to_csv(bam_file, output_folder)
    csv_files.append(csv_file)

# Print the list of newly created CSV files.
csv_files

## 8.3 Filter Alignments by MAPQ Score

In [ ]:
import os
import pandas as pd
from pathlib import Path

# ===== Settings =====
input_dir = Path("fastq/stage1/3_align_bam/csv") # Input folder containing CSV files
output_dir = input_dir / "MAPQ_removed"  # Output folder for filtered CSV files
output_dir.mkdir(parents=True, exist_ok=True)

MAPQ_THRESHOLD = 15     # Keep rows where MAPQ > this value
KEEP_NAN = True         # Keep rows with NaN MAPQ values (e.g., unaligned reads)
# ====================

def process_one_csv(in_path: Path, out_dir: Path, mapq_threshold: int, keep_nan: bool = True):
    out_path = out_dir / in_path.name

    # Remove existing output file to avoid duplicates
    if out_path.exists():
        out_path.unlink()

    # Read input CSV
    try:
        df = pd.read_csv(in_path)
    except Exception as e:
        print(f"⚠️  Read fail: {in_path.name} -> {e}")
        return

    # Skip if MAPQ column does not exist
    if "MAPQ" not in df.columns:
        print(f"⚠️  Skip (no MAPQ column): {in_path.name}")
        return

    # Convert MAPQ column to numeric (invalid entries become NaN)
    m = pd.to_numeric(df["MAPQ"], errors="coerce")

    # Filtering mask: keep MAPQ > threshold, optionally keep NaN
    keep_mask = (m > mapq_threshold) | (m.isna() if keep_nan else False)

    kept = int(keep_mask.sum())
    removed = int((~keep_mask).sum())

    # Save filtered CSV
    df.loc[keep_mask].to_csv(out_path, index=False)
    print(
        f"✅ {in_path.name} → {out_path.name} | kept={kept}, removed={removed} "
        f"| threshold={mapq_threshold}, keep_nan={keep_nan}"
    )

def main():
    csv_files = sorted(input_dir.glob("*.csv"))
    if not csv_files:
        print(f"⚠️  No CSV files in {input_dir}")
        return

    for p in csv_files:
        process_one_csv(p, output_dir, MAPQ_THRESHOLD, KEEP_NAN)

if __name__ == "__main__":
    main()

# Data Analysis

## A. Generate Histogram Data from Aligned Reads(MAPQ filtered)

In [ ]:
import os
import pandas as pd

# Folder setup
input_folder = "fastq/stage1/3_align_bam/csv/MAPQ_removed"
histogram_folder = "fastq/stage1/4_histogram"
os.makedirs(histogram_folder, exist_ok=True)

# Process all CSV files in the input folder
files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]

for file_name in files:
    file_path = os.path.join(input_folder, file_name)
    output_csv = os.path.join(histogram_folder, f"histogram_{file_name}")

    try:
        df = pd.read_csv(file_path, dtype=str)
        if 'RNAME' not in df.columns:
            print(f"Skipping file: {file_name} (no 'RNAME' column found)")
            continue

        # Count the occurrences of each unique RNAME
        rname_counts = df['RNAME'].value_counts().reset_index()
        rname_counts.columns = ['RNAME', 'Count']
        
        # Add metadata and calculate normalized counts
        rname_counts.insert(0, 'File_Name', file_name)
        rname_counts['Count'] = rname_counts['Count'].astype(int)
        total_count = rname_counts['Count'].sum()
        rname_counts['Normalized_Count'] = rname_counts['Count'] / total_count

        # Save the histogram data to a new CSV file
        rname_counts.to_csv(output_csv, index=False)
        print(f"✅ Saved full RNAME histogram: {output_csv}")

    except Exception as e:
        print(f"❌ Error processing file '{file_name}': {e}")

## B. Summarize Highlighted Read Counts into a CSV File

In [ ]:
import os
import re
import pandas as pd
from pathlib import Path

# === Highlight Mapping (Reflecting the latest data format) ===
# Maps specific tags to their corresponding canonical sequence names
highlight_mapping = {
    "0N": "seq_013_00001101",
    "1D": "seq_035_00100011",
    "2S": "seq_082_01010010",
    "3SP": "seq_122_01111010",
    "4G": "seq_134_10000110",
    "5I": "seq_168_10101000",
    "6S": "seq_210_11010010",
    "7T": "seq_243_11110011"
}

# === Folders ===
histogram_folder = Path("fastq/stage1/4_histogram")
summary_folder = Path("fastq/stage1/5_summary")
summary_folder.mkdir(parents=True, exist_ok=True)
highlight_result_csv = summary_folder / "highlight_result.csv"

# === Helpers ===
def canonicalize_rname(x: str) -> str:
    """Standardizes RNAME format to seq_000_00000000"""
    s = str(x).strip()
    m = re.search(r"seq_(\d+)_([01]+)", s)
    if not m: return s
    idx = int(m.group(1))
    bits = m.group(2)
    return f"seq_{idx:03d}_{bits}" # Ensures 3-digit padding

# Standardize the mapping dictionary
normalized_mapping = {k: canonicalize_rname(v) for k, v in highlight_mapping.items()}

def extract_prefix_from_filename(filename: str) -> str:
    """
    Extracts group keywords (e.g., 4G, 4G26) from the filename
    Example: 'histogram_DNA_Data_4G26_N124_assemble.csv' -> '4G26'
    """
    name = filename.replace("histogram_", "").replace(".csv", "")
    tokens = name.split("_")
    
    # Locate the position of 'ID' keyword and analyze preceding tokens
    try:
        idx = tokens.index("ID")
    except ValueError:
        idx = len(tokens)
        
    # Traverse backwards to select the meaningful token (skipping 'N124' or 'DATA')
    j = idx - 1
    while j >= 0:
        token = tokens[j].upper()
        if not re.fullmatch(r"N\d+", token) and token != "DATA":
            return token
        j -= 1
    return ""

# === Main Analysis Loop ===
csv_files = sorted([f for f in os.listdir(histogram_folder) if f.endswith(".csv")])
highlight_rows = []

print(f"🔎 Analyzing a total of {len(csv_files)} histogram files.")

for file in csv_files:
    file_path = histogram_folder / file
    try:
        df = pd.read_csv(file_path)
        if df.empty: continue

        # Column Standardization
        df["RNAME"] = df["RNAME"].apply(canonicalize_rname)
        df["Count"] = pd.to_numeric(df["Count"], errors="coerce").fillna(0).astype(int)
        
        total_count = int(df["Count"].sum())
        if total_count == 0: continue

        # Extract Top 1/2/3 sequences
        top_df = df.sort_values(["Count", "RNAME"], ascending=[False, True]).head(3).reset_index(drop=True)

        def get_top_info(i):
            if i < len(top_df):
                return top_df.loc[i, "RNAME"], int(top_df

## C. Length Analysis(merge length)

In [ ]:
import pd as pd
import os
import re
import numpy as np

# --- 1. Set Answer Key ---
# Mapping category codes to their target 8-bit sequences
answer_data = {
    "0N": "seq_013_00001101",
    "1D": "seq_035_00100011",
    "2S": "seq_082_01010010",
    "3SP": "seq_122_01111010",
    "4G": "seq_134_10000110",
    "5I": "seq_168_10101000",
    "6S": "seq_210_11010010",
    "7T": "seq_243_11110011"
}

# Extract the last 8 binary digits as the reference answer
answer_key_map = {key: re.search(r'([01]{8})$', value).group(1) for key, value in answer_data.items()}

# --- 2. File Processing and Combined Calculation ---
input_folder = "fastq/stage1/4_histogram"
output_folder = "fastq/stage1/5_summary"
output_path = os.path.join(output_folder, "summary_combined.csv")

os.makedirs(output_folder, exist_ok=True)

try:
    # Filter files that follow the histogram naming convention
    files = [f for f in os.listdir(input_folder) if f.endswith('.csv') and f.startswith("histogram_")]
except FileNotFoundError:
    print(f"❌ Error: Folder '{input_folder}' not found.")
    files = []

combined_summary_list = []

if not files:
    print("⚠️ No files found for analysis.")
else:
    print(f"📂 Analyzing a total of {len(files)} files.")
    for file_name in sorted(files):
        try:
            df = pd.read_csv(os.path.join(input_folder, file_name))
            total_count = df['Count'].sum()
            
            # Initialize counts for '0' and '1' at each of the 8 bit positions
            position_counts = [{'0': 0, '1': 0} for _ in range(8)]
            for _, row in df.iterrows():
                rname = row.get('RNAME', '')
                count = int(row['Count'])
                # Extract the 8-bit binary string from RNAME
                match = re.search(r'seq_[^_]+_([01]{8})', str(rname))
                if match:
                    eight_digits = match.group(1)
                    for i, digit in enumerate(eight_digits):
                        position_counts[i][digit] += count
            
            # Determine the correct answer key based on the file name
            answer_key = None
            for key in answer_key_map:
                if key in file_name:
                    answer_key = answer_key_map[key]
                    break
            
            # --- 🚀 Generate 3 rows per file (Zeros, Ones, and Accuracy) ---
            zeros_row = {"File_Name": file_name, "Total_Count": total_count, "X": "Zeros_Count", "Avg_Accuracy": "X"}
            ones_row = {"File_Name": file_name, "Total_Count": total_count, "X": "Ones_Count", "Avg_Accuracy": "X"}
            acc_row = {"File_Name": file_name, "Total_Count": "X", "X": "Avg_Accuracy"}
            
            accuracies_for_avg = [] 
            
            for i in range(8):
                zeros_count = position_counts[i]['0']
                ones_count = position_counts[i]['1']
                
                # Assign count values to columns '1' to '8'
                zeros_row[str(i+1)] = zeros_count
                ones_row[str(i+1)] = ones_count
                
                accuracy = np.nan
                if answer_key:
                    correct_digit = answer_key[i]
                    current_total = zeros_count + ones_count
                    correct_count = position_counts[i][correct_digit]
                    # Calculate accuracy for the current bit position
                    accuracy = correct_count / current_total if current_total > 0 else 0
                
                # Assign accuracy value to columns '1' to '8'
                acc_row[str(i+1)] = accuracy
                accuracies_for_avg.append(accuracy)

            # Calculate the overall mean accuracy for the current file
            avg_accuracy = np.nanmean(accuracies_for_avg) if accuracies_for_avg else np.nan
            acc_row["Avg_Accuracy"] = avg_accuracy

            # Add the block of 3 rows to the results list
            combined_summary_list.extend([zeros_row, ones_row, acc_row])
            
            if not answer_key:
                print(f"⚠️ Warning: Answer key not found for {file_name}.")
            print(f"✅ {file_name} processing complete.")
        
        except Exception as e:
            print(f"❌ Error processing {file_name}: {e}")

# --- 3. Save Combined Results ---
if combined_summary_list:
    final_df = pd.DataFrame(combined_summary_list)
    
    # Define the exact column order for the final CSV
    column_order = ['File_Name', 'Total_Count', 'X', '1', '2', '3', '4', '5', '6', '7', '8', 'Avg_Accuracy']
    final_df = final_df[column_order]

    final_df.to_csv(output_path, index=False)
    
    print("\n--- Final Combined Analysis Results ---")
    print(final_df.head(6).to_string()) # Preview top 6 rows (results for the first 2 files)
    print(f"\n📄 All analysis results have been saved to '{output_path}'.")